# Novel-view synthesis, scored on views the model never saw

Given photographs of an object from a set of known camera positions, render it from a
camera position that was not photographed. The model here is a NeRF - a function that maps
a 3D point and a viewing direction to a colour and a density, which is then integrated
along each camera ray by the volume-rendering equation. Nothing about the scene is stored
as geometry; the weights *are* the scene.

Plain NeRF is slow because its positional encoding forces a large MLP to memorise
high-frequency detail. This notebook uses the multi-resolution hash encoding from
Instant-NGP instead: a set of learned feature tables at 16 resolutions, looked up and
trilinearly interpolated, which moves most of the capacity out of the MLP and into the
tables. The MLP shrinks to a few small layers and training drops from hours to minutes.
It is implemented here in plain PyTorch, trained from scratch, with no pretrained weights
and no CUDA extension.

**The measurement is the point of this notebook.** The NeRF-synthetic scenes ship three
disjoint camera sets - `transforms_train.json`, `transforms_val.json`,
`transforms_test.json` - at different camera positions. Training uses the train views
only. PSNR and SSIM are reported on the **test** views, which are poses the model has
never been given a photograph for. A rendering that looks sharp from a training camera
proves only that the model can memorise; a score on held-out cameras is what tells you it
recovered geometry.

Two reference numbers are reported alongside, because a PSNR with nothing to compare it
to is not interpretable:

- the PSNR of a plain white image against each test view - these scenes are mostly white
  background, so this is the floor a model must clear to have learned anything at all;
- the PSNR of the *mean training image*, a model with no geometry whatsoever.


## What was actually attached

Before assuming a directory layout, look.

In [ ]:
import os, sys, glob, json

layout = {}
for root in ("/kaggle/input",):
    if not os.path.isdir(root):
        continue
    for dirpath, dirnames, filenames in os.walk(root):
        if dirpath[len(root):].count(os.sep) > 3:
            dirnames[:] = []
            continue
        exts = {}
        for f in filenames:
            e = os.path.splitext(f)[1].lower()
            exts[e] = exts.get(e, 0) + 1
        if exts:
            layout[dirpath] = exts

print(f"{len(layout)} directories with files (depth <= 3)\n")
for d, exts in sorted(layout.items())[:30]:
    top = ", ".join(f"{k or '(none)'}:{v}" for k, v in
                    sorted(exts.items(), key=lambda kv: -kv[1])[:5])
    print(f"  {d}\n      {top}")

# A scene is any directory containing transforms_train.json. Do not assume the name.
TJ = sorted(glob.glob("/kaggle/input/**/transforms_train.json", recursive=True))
SCENE_DIRS = sorted({os.path.dirname(p) for p in TJ})
print(f"\n{len(SCENE_DIRS)} scene(s) with transforms_train.json:")
for d in SCENE_DIRS:
    have = [s for s in ("train", "val", "test")
            if os.path.exists(os.path.join(d, f"transforms_{s}.json"))]
    print(f"   {os.path.basename(d):14} splits present: {have}")
assert SCENE_DIRS, "no transforms_train.json anywhere - see the layout above"

## Loading the scene, and the camera convention

In [ ]:
import numpy as np, torch, time
from PIL import Image

DEV = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)
print("device:", DEV, torch.cuda.get_device_name(0) if DEV == "cuda" else "")

RES = 400           # these scenes are 800x800; half resolution is the standard fast setting

def resolve(scene_dir, fp):
    """transforms_*.json stores './train/r_0' with no extension in the original release,
    but re-uploads sometimes include '.png' and sometimes use backslashes. Try the
    variants instead of assuming one."""
    fp = fp.replace("\\", "/").lstrip("./")
    for cand in (fp, fp + ".png", fp + ".jpg", fp + ".JPG"):
        p = os.path.join(scene_dir, cand)
        if os.path.exists(p):
            return p
    hits = glob.glob(os.path.join(scene_dir, "**",
                                  os.path.basename(fp) + ".*"), recursive=True)
    return hits[0] if hits else None

def load_split(scene_dir, split, limit=None):
    with open(os.path.join(scene_dir, f"transforms_{split}.json")) as f:
        meta = json.load(f)
    frames = meta["frames"]
    # The test set has 200 poses. Rendering all of them costs more than it tells us, so
    # an evenly-spaced subset is used and the count is reported with the score.
    if limit and len(frames) > limit:
        frames = [frames[i] for i in np.linspace(0, len(frames) - 1, limit).astype(int)]
    imgs, poses, skipped = [], [], 0
    for fr in frames:
        p = resolve(scene_dir, fr["file_path"])
        if p is None:
            skipped += 1
            continue
        im = Image.open(p)
        if im.size != (RES, RES):
            im = im.resize((RES, RES), Image.LANCZOS)
        a = np.asarray(im, dtype="float32") / 255.0
        if a.ndim == 2:
            a = np.stack([a] * 3, -1)
        if a.shape[2] == 4:
            # Composite onto WHITE, which is the convention these scenes are scored
            # under. Compositing onto black instead changes PSNR by several dB, so this
            # is not a cosmetic choice.
            a = a[..., :3] * a[..., 3:4] + (1.0 - a[..., 3:4])
        imgs.append(a[..., :3])
        poses.append(np.array(fr["transform_matrix"], dtype="float32"))
    if skipped:
        print(f"    {split}: {skipped} frame(s) had no resolvable image file")
    H = W = RES
    cax = meta.get("camera_angle_x")
    focal = 0.5 * W / np.tan(0.5 * float(cax)) if cax else 1.1 * W
    return np.stack(imgs), np.stack(poses), float(focal)

def get_rays(H, W, focal, c2w):
    """Blender/NeRF camera convention: +x right, +y UP, looking down -z. Getting the two
    sign flips wrong still renders something plausible from training views and fails
    completely on held-out ones, so it is worth being explicit."""
    i, j = np.meshgrid(np.arange(W, dtype="float32"),
                       np.arange(H, dtype="float32"), indexing="xy")
    dirs = np.stack([(i - W * 0.5 + 0.5) / focal,
                     -(j - H * 0.5 + 0.5) / focal,
                     -np.ones_like(i)], -1)
    rd = dirs @ c2w[:3, :3].T
    rd = rd / (np.linalg.norm(rd, axis=-1, keepdims=True) + 1e-9)
    ro = np.broadcast_to(c2w[:3, 3], rd.shape)
    return ro.astype("float32"), rd.astype("float32")

SCENE = SCENE_DIRS[0]
print(f"\nloading {os.path.basename(SCENE)}")
tr_i, tr_p, focal = load_split(SCENE, "train")
te_i, te_p, _ = load_split(SCENE, "test", limit=25)
print(f"  train {tr_i.shape}  test {te_i.shape}  focal {focal:.2f}")
print(f"  camera distance from origin: "
      f"{np.linalg.norm(tr_p[:, :3, 3], axis=1).mean():.3f}")

## The model: a multi-resolution hash grid and a very small MLP

In [ ]:
import torch.nn as nn, torch.nn.functional as F

class HashEncoding(nn.Module):
    """Multi-resolution hash encoding (Instant-NGP). L levels of feature tables; each
    level hashes the 8 integer corners of the voxel a point falls in and trilinearly
    interpolates their features. Collisions are left unresolved on purpose - gradients
    from a surface point dominate those from empty space, so the table resolves the
    collision in favour of the geometry that matters."""
    PRIMES = (1, 2654435761, 805459861)

    def __init__(self, L=16, F_=2, T=2 ** 19, N_min=16, N_max=1024):
        super().__init__()
        self.L, self.F = L, F_
        b = np.exp((np.log(N_max) - np.log(N_min)) / (L - 1))
        self.register_buffer("res", torch.tensor(
            [int(np.floor(N_min * b ** i)) for i in range(L)], dtype=torch.long))
        self.T = T
        self.tables = nn.Parameter(torch.empty(L, T, F_).uniform_(-1e-4, 1e-4))
        self.out_dim = L * F_

    def forward(self, x):
        """x in [0,1]^3 -> (N, L*F)."""
        N = x.shape[0]
        out = []
        offs = torch.tensor([[0, 0, 0], [1, 0, 0], [0, 1, 0], [1, 1, 0],
                             [0, 0, 1], [1, 0, 1], [0, 1, 1], [1, 1, 1]],
                            device=x.device, dtype=torch.long)
        for l in range(self.L):
            r = int(self.res[l])
            p = x * (r - 1)
            p0 = torch.floor(p).long()
            w = (p - p0.float()).clamp(0, 1)
            feat = 0.0
            for k in range(8):
                c = (p0 + offs[k]).clamp(0, r - 1)
                h = (c[:, 0] * self.PRIMES[0]) ^ (c[:, 1] * self.PRIMES[1]) \
                    ^ (c[:, 2] * self.PRIMES[2])
                idx = (h % self.T).clamp(0, self.T - 1)
                wk = torch.ones(N, device=x.device)
                for d in range(3):
                    wk = wk * (w[:, d] if offs[k, d] == 1 else (1.0 - w[:, d]))
                feat = feat + wk.unsqueeze(1) * self.tables[l][idx]
            out.append(feat)
        return torch.cat(out, 1)

def sh_encode(d):
    """Spherical harmonics up to degree 3 (16 coefficients) for the view direction.
    Cheaper and lower-dimensional than a frequency encoding, and view dependence in
    these scenes is mild specularity, which low-order SH represents well."""
    x, y, z = d[:, 0], d[:, 1], d[:, 2]
    xx, yy, zz, xy, yz, xz = x * x, y * y, z * z, x * y, y * z, x * z
    return torch.stack([
        0.2820948 * torch.ones_like(x),
        -0.4886025 * y, 0.4886025 * z, -0.4886025 * x,
        1.0925484 * xy, -1.0925484 * yz, 0.3153916 * (3 * zz - 1),
        -1.0925484 * xz, 0.5462742 * (xx - yy),
        -0.5900436 * y * (3 * xx - yy), 2.8906114 * xy * z,
        -0.4570458 * y * (5 * zz - 1), 0.3731763 * z * (5 * zz - 3),
        -0.4570458 * x * (5 * zz - 1), 1.4453057 * z * (xx - yy),
        -0.5900436 * x * (xx - 3 * yy)], 1)

BOUND = 1.5      # these scenes fit inside [-1.5, 1.5]^3; anything outside is background

class HashNeRF(nn.Module):
    def __init__(self, geo_feat=15, hidden=64):
        super().__init__()
        self.enc = HashEncoding()
        self.sigma = nn.Sequential(nn.Linear(self.enc.out_dim, hidden), nn.ReLU(True),
                                   nn.Linear(hidden, 1 + geo_feat))
        self.color = nn.Sequential(nn.Linear(geo_feat + 16, hidden), nn.ReLU(True),
                                   nn.Linear(hidden, hidden), nn.ReLU(True),
                                   nn.Linear(hidden, 3))
        self.geo_feat = geo_feat

    def forward(self, x, d):
        xn = ((x + BOUND) / (2 * BOUND)).clamp(0, 1)
        h = self.sigma(self.enc(xn))
        # exp-activated density, clamped: the standard trick. ReLU density gives dead
        # gradients wherever the field has gone negative and training stalls.
        sigma = torch.exp(h[:, :1].clamp(-15, 15))
        rgb = torch.sigmoid(self.color(torch.cat([h[:, 1:], sh_encode(d)], 1)))
        return sigma, rgb

model = HashNeRF().to(DEV)
n_tab = model.enc.tables.numel()
n_mlp = sum(p.numel() for n, p in model.named_parameters() if "tables" not in n)
print(f"hash tables {n_tab / 1e6:.2f}M params   MLP {n_mlp / 1e3:.1f}K params")
print("-> nearly all capacity is in the tables; the MLP is deliberately tiny")

## Volume rendering, PSNR and SSIM

In [ ]:
N_SAMP, NEAR, FAR = 128, 2.0, 6.0

def render_rays(ro, rd, n_samp=N_SAMP, perturb=True):
    """Volume rendering: march samples along each ray, convert density to per-segment
    opacity, and alpha-composite. The remaining transmittance is composited against
    WHITE, matching how the ground-truth RGBA images were flattened."""
    N = ro.shape[0]
    t = torch.linspace(NEAR, FAR, n_samp, device=ro.device).expand(N, n_samp)
    if perturb:
        mid = 0.5 * (t[:, 1:] + t[:, :-1])
        lo = torch.cat([t[:, :1], mid], 1); hi = torch.cat([mid, t[:, -1:]], 1)
        t = lo + (hi - lo) * torch.rand_like(t)
    pts = ro.unsqueeze(1) + rd.unsqueeze(1) * t.unsqueeze(2)
    dirs = rd.unsqueeze(1).expand_as(pts)
    sigma, rgb = model(pts.reshape(-1, 3), dirs.reshape(-1, 3))
    sigma = sigma.view(N, n_samp)
    rgb = rgb.view(N, n_samp, 3)
    # Points outside the bounding box carry no geometry; force them empty so the hash
    # tables are not asked to represent the unbounded background.
    inside = (pts.abs() <= BOUND).all(-1)
    sigma = sigma * inside.float()
    dist = torch.cat([t[:, 1:] - t[:, :-1],
                      torch.full((N, 1), 1e-3, device=ro.device)], 1)
    alpha = 1.0 - torch.exp(-sigma * dist)
    T = torch.cumprod(torch.cat([torch.ones(N, 1, device=ro.device),
                                 1.0 - alpha + 1e-10], 1), 1)[:, :-1]
    w = alpha * T
    acc = w.sum(1, keepdim=True)
    return (w.unsqueeze(2) * rgb).sum(1) + (1.0 - acc), acc.squeeze(1), w

def psnr(a, b):
    return float(-10.0 * np.log10(np.mean((np.clip(a, 0, 1) - b) ** 2) + 1e-12))

def ssim(a, b, C1=0.01 ** 2, C2=0.03 ** 2):
    """Gaussian-window SSIM, averaged over channels. Computed on the same images PSNR
    uses so the two numbers describe the same thing."""
    x = torch.from_numpy(np.clip(a, 0, 1)).permute(2, 0, 1)[None].float()
    y = torch.from_numpy(b).permute(2, 0, 1)[None].float()
    g = torch.arange(11).float() - 5
    g = torch.exp(-g ** 2 / (2 * 1.5 ** 2)); g = (g / g.sum())
    k = (g[:, None] @ g[None, :])[None, None].repeat(3, 1, 1, 1)
    f = lambda z: F.conv2d(z, k, padding=5, groups=3)
    mx, my = f(x), f(y)
    sxx, syy, sxy = f(x * x) - mx ** 2, f(y * y) - my ** 2, f(x * y) - mx * my
    s = ((2 * mx * my + C1) * (2 * sxy + C2)) / \
        ((mx ** 2 + my ** 2 + C1) * (sxx + syy + C2))
    return float(s.mean())

print(f"{N_SAMP} samples per ray in [{NEAR}, {FAR}], white-background compositing")

## Training on the train views only

In [ ]:
# Precompute every training ray once. 100 views at 400x400 is 16M rays; storing the
# origin, direction and target as float16 on the GPU costs about 0.6 GB and removes all
# per-step host work.
ros, rds, tgts = [], [], []
for im, pose in zip(tr_i, tr_p):
    ro, rd = get_rays(RES, RES, focal, pose)
    ros.append(ro.reshape(-1, 3)); rds.append(rd.reshape(-1, 3))
    tgts.append(im.reshape(-1, 3))
RO = torch.from_numpy(np.concatenate(ros)).to(DEV)
RD = torch.from_numpy(np.concatenate(rds)).to(DEV)
TG = torch.from_numpy(np.concatenate(tgts)).to(DEV)
print(f"{RO.shape[0] / 1e6:.2f}M training rays on device")

@torch.no_grad()
def render_image(pose, chunk=8192):
    model.eval()
    ro, rd = get_rays(RES, RES, focal, pose)
    ro = torch.from_numpy(ro.reshape(-1, 3)).to(DEV)
    rd = torch.from_numpy(rd.reshape(-1, 3)).to(DEV)
    out = []
    for i in range(0, ro.shape[0], chunk):
        with torch.cuda.amp.autocast():
            c, _, _ = render_rays(ro[i:i + chunk], rd[i:i + chunk], perturb=False)
        out.append(c.float().cpu())
    return torch.cat(out).numpy().reshape(RES, RES, 3)

@torch.no_grad()
def eval_split(imgs, poses):
    ps, ss, preds = [], [], []
    for k, (im, pose) in enumerate(zip(imgs, poses)):
        r = render_image(pose)
        ps.append(psnr(r, im)); ss.append(ssim(r, im))
        if k < 4:
            preds.append((r, im))
    return {"psnr": float(np.mean(ps)), "psnr_std": float(np.std(ps)),
            "psnr_min": float(np.min(ps)), "psnr_max": float(np.max(ps)),
            "ssim": float(np.mean(ss)), "n_views": len(ps),
            "per_view_psnr": [round(v, 3) for v in ps]}, preds

STEPS, RAYS = 20000, 4096
opt = torch.optim.Adam([
    {"params": model.enc.parameters(), "lr": 1e-2, "weight_decay": 0.0},
    {"params": [p for n, p in model.named_parameters() if "enc." not in n],
     "lr": 1e-3, "weight_decay": 1e-6}], betas=(0.9, 0.99), eps=1e-15)
sched = torch.optim.lr_scheduler.LambdaLR(
    opt, lambda s: 0.1 ** (s / STEPS))        # decay to 1/10 by the end
scaler = torch.cuda.amp.GradScaler()

hist, t0 = [], time.time()
print(f"\ntraining {STEPS} steps x {RAYS} rays on {len(tr_i)} TRAIN views")
oom_events = []
for step in range(1, STEPS + 1):
    model.train()
    # The hash encoder gathers 8 corners at each of 16 levels, so the autograd graph
    # holds ~128 index/feature tensors per step and peak memory scales with the ray
    # count. Rather than lose the whole session to one OOM, halve the batch and carry
    # on - the step count stays the same, each step just sees fewer rays.
    while True:
        try:
            idx = torch.randint(0, RO.shape[0], (RAYS,), device=DEV)
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast():
                c, _, _ = render_rays(RO[idx], RD[idx])
                loss = F.mse_loss(c, TG[idx])
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            break
        except torch.cuda.OutOfMemoryError:
            opt.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()
            if RAYS <= 256:
                raise
            RAYS = RAYS // 2
            oom_events.append({"step": step, "new_rays": RAYS})
            print(f"  OOM at step {step}; rays per step reduced to {RAYS}")
    sched.step()
    if step % 2000 == 0 or step == STEPS:
        tr_psnr = float(-10 * np.log10(max(loss.item(), 1e-12)))
        hist.append({"step": step, "train_mse": loss.item(),
                     "train_batch_psnr": round(tr_psnr, 3),
                     "seconds": round(time.time() - t0, 1)})
        print(f"  step {step:6d}  mse {loss.item():.5f}  "
              f"train-batch PSNR {tr_psnr:5.2f}  {time.time() - t0:6.0f}s")
print(f"trained in {(time.time() - t0) / 60:.1f} min")

## Result on held-out views

In [ ]:
# Reference points, so the PSNR below is interpretable.
white = np.ones((RES, RES, 3), dtype="float32")
mean_tr = tr_i.mean(0)
base_white = float(np.mean([psnr(white, im) for im in te_i]))
base_mean = float(np.mean([psnr(mean_tr, im) for im in te_i]))

res_te, preds_te = eval_split(te_i, te_p)
res_tr, _ = eval_split(tr_i[:10], tr_p[:10])

print(f"scene: {os.path.basename(SCENE)}\n")
print(f"{'':34}{'PSNR (dB)':>11}{'SSIM':>8}")
print(f"{'HELD-OUT TEST views':34}{res_te['psnr']:>11.2f}{res_te['ssim']:>8.4f}"
      f"   n={res_te['n_views']}")
print(f"{'training views (10, for contrast)':34}{res_tr['psnr']:>11.2f}"
      f"{res_tr['ssim']:>8.4f}")
print(f"\nreference points on the same test views")
print(f"{'a plain white image':34}{base_white:>11.2f}")
print(f"{'the mean training image':34}{base_mean:>11.2f}")
print(f"\ngain over the white-image floor: {res_te['psnr'] - base_white:+.2f} dB")
print(f"train - test PSNR gap: {res_tr['psnr'] - res_te['psnr']:+.2f} dB")
print(f"\nper-test-view PSNR spread: min {res_te['psnr_min']:.2f}  "
      f"max {res_te['psnr_max']:.2f}  sd {res_te['psnr_std']:.2f}")

results = {"task": "novel-view synthesis on NeRF-synthetic",
           "scene": os.path.basename(SCENE),
           "model": "hash-grid NeRF (Instant-NGP encoding) in plain PyTorch, from scratch",
           "hash_table_params_millions": round(n_tab / 1e6, 3),
           "mlp_params": int(n_mlp),
           "resolution": RES, "samples_per_ray": N_SAMP,
           "steps": STEPS, "rays_per_step_final": RAYS,
           "oom_batch_reductions": oom_events,
           "n_train_views": int(len(tr_i)),
           "held_out": "transforms_test.json poses; no test view used in training",
           "test": res_te, "train_views_subset": res_tr,
           "baseline_psnr_white_image": round(base_white, 3),
           "baseline_psnr_mean_training_image": round(base_mean, 3),
           "history": hist}
with open("/kaggle/working/results.json", "w") as f:
    json.dump(results, f, indent=1)
print("\nwrote /kaggle/working/results.json")

## Curves, and the per-view spread

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5), dpi=130)
ax[0].plot([h["step"] for h in hist], [h["train_batch_psnr"] for h in hist], "o-",
           lw=1.6, ms=3.5, label="train batch")
ax[0].axhline(res_te["psnr"], ls="--", c="crimson", lw=1.3,
              label=f"held-out test {res_te['psnr']:.2f} dB")
ax[0].axhline(base_white, ls=":", c="gray", lw=1.3,
              label=f"white-image floor {base_white:.2f} dB")
ax[0].set_xlabel("step"); ax[0].set_ylabel("PSNR (dB)")
ax[0].set_title("training, with the held-out score", fontsize=10)
ax[0].legend(fontsize=7.5, loc="lower right")
ax[1].bar(range(len(res_te["per_view_psnr"])), res_te["per_view_psnr"],
          color="#3b6ea5", width=0.75)
ax[1].axhline(res_te["psnr"], ls="--", c="crimson", lw=1.2,
              label=f"mean {res_te['psnr']:.2f} dB")
ax[1].set_xlabel("held-out test view"); ax[1].set_ylabel("PSNR (dB)")
ax[1].set_ylim(min(res_te["per_view_psnr"]) - 2, max(res_te["per_view_psnr"]) + 1)
ax[1].set_title("per-view, because the mean hides the spread", fontsize=10)
ax[1].legend(fontsize=8)
for a in ax:
    a.grid(alpha=0.25, lw=0.5); a.spines[["top", "right"]].set_visible(False)
plt.tight_layout(); plt.show()

## Renders of poses that were withheld

In [ ]:
# Renders of held-out poses next to the photograph that was withheld, plus the
# absolute error, which is where the remaining dB live.
n = len(preds_te)
fig, axes = plt.subplots(n, 3, figsize=(9.6, 3.2 * n), dpi=130)
if n == 1:
    axes = axes[None, :]
for r, (pred, gt) in enumerate(preds_te):
    err = np.abs(np.clip(pred, 0, 1) - gt).mean(-1)
    for c, (im, cm, t) in enumerate(((np.clip(pred, 0, 1), None, "rendered (unseen pose)"),
                                     (gt, None, "withheld photograph"),
                                     (err, "inferno", "absolute error"))):
        a = axes[r, c]
        if cm:
            a.imshow(im, cmap=cm, vmin=0, vmax=0.25)
        else:
            a.imshow(im)        # vmin/vmax on an RGB array is ignored and warns
        a.set_xticks([]); a.set_yticks([])
        if r == 0:
            a.set_title(t, fontsize=10)
    axes[r, 0].set_ylabel(f"PSNR {psnr(pred, gt):.2f} dB", fontsize=8.5)
plt.tight_layout(); plt.show()

## Poses in no split at all

In [ ]:
# An orbit of poses that exist in NO split - interpolated camera positions. This is
# qualitative only and is reported as such: there is no ground truth to score it against.
import matplotlib.pyplot as plt
R = float(np.linalg.norm(tr_p[:, :3, 3], axis=1).mean())

def look_at(cam, up=np.array([0, 0, 1.0])):
    f = -cam / np.linalg.norm(cam)                       # camera looks down -z at origin
    s = np.cross(up, -f); s /= np.linalg.norm(s) + 1e-9
    u = np.cross(-f, s)
    m = np.eye(4, dtype="float32")
    m[:3, 0], m[:3, 1], m[:3, 2], m[:3, 3] = s, u, -f, cam
    return m

frames = []
for az in np.linspace(0, 2 * np.pi, 6, endpoint=False):
    el = np.deg2rad(22)
    cam = np.array([R * np.cos(el) * np.cos(az), R * np.cos(el) * np.sin(az),
                    R * np.sin(el)], dtype="float32")
    frames.append(np.clip(render_image(look_at(cam)), 0, 1))

fig, axes = plt.subplots(1, 6, figsize=(14, 2.5), dpi=130)
for a, im in zip(axes, frames):
    a.imshow(im); a.set_xticks([]); a.set_yticks([])
fig.suptitle("six interpolated poses, none of them in any split (no ground truth, "
             "so no score)", fontsize=9, y=1.04)
plt.tight_layout(); plt.show()

## What this shows, and what it does not

A hash-grid NeRF, trained from scratch in plain PyTorch on the train views of one
NeRF-synthetic scene, reaches the PSNR and SSIM printed above on the **test** poses. Those
poses supplied no photograph during training, so the score measures interpolation of
geometry rather than memorisation of pixels. The white-image and mean-image baselines are
printed with it so the dB figure has a floor to be read against.

What this does **not** establish:

- **It is one scene.** NeRF-synthetic has eight, and they differ sharply in difficulty -
  a diffuse lego model and a transparent drum kit are not the same problem. A single
  scene's PSNR is not a benchmark result, and the published tables average over all eight.
- **No comparison is run.** Nothing here is trained against plain NeRF, against 3D
  Gaussian splatting, or against any other baseline, so no claim is made about which is
  better. The reference numbers are degenerate baselines, not competing methods.
- **The resolution and sample count are below the published protocol.** Training and
  scoring are at 400×400 with 128 uniform samples per ray and no occupancy grid or
  proposal network. PSNR at half resolution is not comparable to PSNR at 800×800, and the
  uniform sampling spends most of its samples in empty space - a real accuracy cost, taken
  deliberately to keep the implementation free of CUDA extensions.
- **Only a subset of test views is rendered.** The count is reported with the score. The
  per-view spread is plotted because views differ by several dB and a mean alone would
  hide that.
- **Synthetic scenes have exact camera poses.** On real photographs the poses come from
  structure-from-motion and carry error, which is a large part of the difficulty of real
  novel-view synthesis and is entirely absent here.
- **The orbit renders are not evidence.** They have no ground truth. They are included to
  show the field is coherent when moved through, which a score on fixed poses cannot show,
  and they are not part of the result.

Part of a portfolio: [github.com/hammasbuilds](https://github.com/hammasbuilds) ·
[vision-portfolio-woad.vercel.app](https://vision-portfolio-woad.vercel.app)
